In [ ]:
import cv2
import numpy as np
import time
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

# --- PASO 1: CARGA EXPLÍCITA UNA POR UNA ---
def cargar_imagenes_manual():
    X = []
    y = []

    # 1. Cargar las 11 imágenes obscuras
    print("Leyendo imágenes obscuras...")
    for i in range(1, 12):
        ruta = f"/content/Imagenes obscuras/Imagen{i}.png"
        if os.path.exists(ruta):
            img = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, (64, 64))
                X.append(img)
                y.append(0) # 0 = Obscura
                print(f"✅ OK: Imagen{i}.png")
        else:
            print(f"⚠️ No encontrada: {ruta}")

    # 2. Cargar las 12 imágenes claras
    print("\nLeyendo imágenes claras...")
    for i in range(1, 13):
        # Ajusta aquí si tus claras empiezan en otro número (ej. Imagen12)
        ruta = f"/content/Imagenes claras/Imagen{i}.png"
        if os.path.exists(ruta):
            img = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                img = cv2.resize(img, (64, 64))
                X.append(img)
                y.append(1) # 1 = Clara
                print(f"✅ OK: Imagen{i}.png")
        else:
            print(f"⚠️ No encontrada: {ruta}")

    return np.array(X), np.array(y)

# Procesar los datos
X_data, y_data = cargar_imagenes_manual()

if len(X_data) < 2:
    print("❌ Error: No se cargaron suficientes imágenes. Revisa las rutas.")
else:
    # Normalizar y preparar
    X = X_data / 255.0
    X = X.reshape(-1, 64, 64, 1)
    X_train, X_test, y_train, y_test = train_test_split(X, y_data, test_size=0.2, random_state=42)

    # --- PASO 2: PROBAR LAS 4 COMBINACIONES ---
    configs = [
        {"nombre": "1. Simple (16 neuronas)", "capas": [16]},
        {"nombre": "2. Media (64, 32)", "capas": [64, 32]},
        {"nombre": "3. Profunda (128, 64, 32)", "capas": [128, 64, 32]},
        {"nombre": "4. Ancha (512 neuronas)", "capas": [512]}
    ]

    resultados = []

    print(f"\nEntrenando {len(X_train)} imágenes...")

    for c in configs:
        # Crear modelo
        model = models.Sequential()
        model.add(layers.Flatten(input_shape=(64, 64, 1)))
        for n in c["capas"]:
            model.add(layers.Dense(n, activation='relu'))
        model.add(layers.Dense(1, activation='sigmoid'))

        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

        # Medir tiempo
        inicio = time.time()
        model.fit(X_train, y_train, epochs=25, batch_size=4, verbose=0)
        tiempo_final = time.time() - inicio

        # Evaluar
        loss, acc = model.evaluate(X_test, y_test, verbose=0)

        resultados.append({
            "Modelo": c["nombre"],
            "Acc": acc,
            "Tiempo": tiempo_final
        })

        # Imprimir resultados de esta iteración
        print(f"\n> {c['nombre']}")
        print(f"  Precisión: {acc*100:.2f}%")
        print(f"  Tiempo: {tiempo_final:.4f} seg")

    # --- PASO 3: SELECCIÓN DEL MEJOR ---
    mejor = max(resultados, key=lambda x: x["Acc"])
    print("\n" + "="*30)
    print(f"🏆 EL MEJOR MODELO FUE: {mejor['Modelo']}")
    print(f"Logró {mejor['Acc']*100:.2f}% de precisión.")
    print("="*30)

Leyendo imágenes obscuras...
✅ OK: Imagen1.png
✅ OK: Imagen2.png
✅ OK: Imagen3.png
✅ OK: Imagen4.png
✅ OK: Imagen5.png
✅ OK: Imagen6.png
✅ OK: Imagen7.png
✅ OK: Imagen8.png
✅ OK: Imagen9.png
✅ OK: Imagen10.png
⚠️ No encontrada: /content/Imagenes obscuras/Imagen11.png

Leyendo imágenes claras...
⚠️ No encontrada: /content/Imagenes claras/Imagen1.png
⚠️ No encontrada: /content/Imagenes claras/Imagen2.png
⚠️ No encontrada: /content/Imagenes claras/Imagen3.png
⚠️ No encontrada: /content/Imagenes claras/Imagen4.png
⚠️ No encontrada: /content/Imagenes claras/Imagen5.png
⚠️ No encontrada: /content/Imagenes claras/Imagen6.png
⚠️ No encontrada: /content/Imagenes claras/Imagen7.png
⚠️ No encontrada: /content/Imagenes claras/Imagen8.png
⚠️ No encontrada: /content/Imagenes claras/Imagen9.png
⚠️ No encontrada: /content/Imagenes claras/Imagen10.png
⚠️ No encontrada: /content/Imagenes claras/Imagen11.png
✅ OK: Imagen12.png

Entrenando 8 imágenes...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



> 1. Simple (16 neuronas)
  Precisión: 100.00%
  Tiempo: 1.8497 seg

> 2. Media (64, 32)
  Precisión: 100.00%
  Tiempo: 1.9600 seg

> 3. Profunda (128, 64, 32)
  Precisión: 100.00%
  Tiempo: 3.0678 seg

> 4. Ancha (512 neuronas)
  Precisión: 100.00%
  Tiempo: 2.5096 seg

🏆 EL MEJOR MODELO FUE: 1. Simple (16 neuronas)
Logró 100.00% de precisión.
